In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import seaborn as sns
from mne.viz import plot_topomap

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from sklearn.decomposition import PCA, FastICA  # noqa: E402

from scripts.analysis_common import (  # noqa: E402
    FREQUENCY_BANDS,
    WAVELET_BAND_FREQ_RESOLUTION_HZ,
    analyzers_to_datasets,
    load_analyzers,
    wavelet_transform,
)
from src.analysis.wavelet_ica import zscore_by_time  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
%matplotlib inline

# Subject–Frequency Features ICA on Wavelet Power

## Scope

This notebook performs ICA decomposition on wavelet power data where
**subjects and frequencies** form the observation axis, and **channels
and time** are combined into the feature axis:

```
Input:   (n_subjects, n_channels, n_freqs, n_times)  — 4-D wavelet power
Reshape: (n_subjects × n_freqs,  n_channels × n_times)
         ──── observations ────  ──── features ────────
```

Before reshaping the tensor is **z-scored along the time axis** so that
every `(subject, channel, frequency)` slice has zero mean and unit
variance.  This removes overall amplitude differences and ensures that
PCA/ICA operates on standardised activations.

## What the decomposition finds

Each observation is a specific **subject–frequency combination**, described
by its full channel × time power surface.  PCA followed by ICA discovers
a small set of **spatial-temporal component patterns** (each of length
C × T) — recurring channel × time patterns shared across subjects and
frequencies.  The ICA score for each component can be reshaped back to
`(n_subjects, n_freqs)` and decomposed into:

| Quantity | Shape | Interpretation |
|----------|-------|----------------|
| **Spatial-temporal component** | `(C, T)` | A channel × time pattern shared across observations |
| **Channel loadings** | `(C,)` | Time-averaged spatial topography (scalp map) |
| **Temporal profile** | `(T,)` | Channel-averaged time course of the component |
| **Subject loadings** | `(S,)` | Mean |score| over frequencies — which participants contribute most |
| **Frequency loadings** | `(F,)` | Mean |score| over subjects — which bands dominate |

## Why this reshape?

By combining subjects and frequencies into observations while keeping
channels and time as features, this decomposition is optimised for
finding **spatial-temporal modes** — network-level activation patterns
that appear across both subjects and frequency bands.  This is
complementary to the combined-features approach (Approach 1), which
finds temporal patterns.

The key advantage is that the observation axis (S × F) groups the data
by *who* and *what frequency*, allowing ICA to discover channel × time
fingerprints that generalise across both individuals and spectral bands.
If a component has high scores for many subjects and frequencies, it
represents a robust, stimulus-driven spatial-temporal mode.

## Analyses

1. Z-scoring and reshape
2. PCA dimensionality reduction + ICA decomposition
3. **(a)** Intersubject correlation matrix of ICA components
4. **(b)** Component temporal profiles — mean ± std across subjects
5. **(c)** Frequency × Time mean-loading heatmaps of component activations
6. **(d)** Mean and variance of component channel loadings as topomaps
7. **(e)** Per-subject loading bar plot for each component
8. **(f)** IC temporal patterns — channel-averaged component waveforms
9. **(g)** Per-subject per-component loading heatmap
10. **(h)** Subject-consistency bar plot per component
11. **(i)** Frequency profile per component

## Configuration


In [ ]:
# ── Experiment configuration ─────────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]  # single type for fast exploration
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# ── Wavelet settings ─────────────────────────────────────────────────────────
REPRESENTATION = "power"
WAVELET_FREQ_MIN = min(lo for lo, _ in FREQUENCY_BANDS.values())
WAVELET_FREQ_MAX = max(hi for _, hi in FREQUENCY_BANDS.values())
WAVELET_N_FREQS = max(
    2,
    int(round((WAVELET_FREQ_MAX - WAVELET_FREQ_MIN) / WAVELET_BAND_FREQ_RESOLUTION_HZ))
    + 1,
)
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_subjects, n_channels, n_freqs, n_times)

# ── Reuse / compute ──────────────────────────────────────────────────────────
REUSE_WAVELETS = True  # load from cache; set False to compute + save

# ── Subject subset ────────────────────────────────────────────────────────────
N_SUBJECTS_SUBSET: int | None = 5

# ── Channel and time subset ───────────────────────────────────────────────────
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 10000  # first N time samples

# ── Decomposition settings ────────────────────────────────────────────────────
N_COMPONENTS_PCA = 50  # number of PCA components to retain
N_COMPONENTS_ICA = 10  # number of ICA components to extract
ICA_RANDOM_STATE = 42  # reproducibility

# ── Storage directory ─────────────────────────────────────────────────────────
WAVELET_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR / "04-wavelet-ica-analysis" / "wavelet_cache"
)

# ── Plot saving ──────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "04-wavelet-ica-analysis"
    / "plots"
    / "subject_freq_features"
    / "pca_ica"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wavelet base directory : {WAVELET_DIR}")
print(
    f"Frequencies            : {FREQS[0]:.1f}\u2013{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)"
)
print(f"PCA components         : {N_COMPONENTS_PCA}")
print(f"ICA components         : {N_COMPONENTS_ICA}")

## Data Loading


In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
    normalize_data=False,
)
datasets = analyzers_to_datasets(analyzers)

# Always limit to first N_SUBJECTS_SUBSET individuals
if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals.")

# Slice to channel subset
if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_CHANNELS_SUBSET} channels.")

# Slice to time subset
if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_TIMES_SUBSET} time samples.")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, "
        f"{ad.n_samples} samples"
    )

## Load or Compute Wavelet Transforms

Stored in `WAVELET_DIR/broadband/`.


In [ ]:
broadband_datasets = wavelet_transform(
    datasets=datasets,
    freqs=FREQS,
    representation=REPRESENTATION,
    keep_frequency_dim=KEEP_FREQUENCY_DIM,
    reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
    wavelet_dir=WAVELET_DIR / "broadband",
    reuse_wavelets=REUSE_WAVELETS,
)
for label, ad in broadband_datasets.items():
    source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
    print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

## Dataset Selection

Change `LABEL` to switch between music types.  The remaining cells use
`bb_data` (broadband wavelet power, 4-D) and derived quantities.


In [ ]:
LABEL = list(broadband_datasets.keys())[0]

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

print(f"Dataset    : {LABEL}")
print(
    f"Shape      : {bb_data.shape}  (subjects \u00d7 channels \u00d7 freqs \u00d7 times)"
)
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}\u2013{FREQS[-1]:.1f} Hz ({n_freqs} steps)")

---
## Step 1 — Z-score and Reshape

**Z-scoring** normalises each `(subject, channel, frequency)` time series
to zero mean and unit variance.  This ensures that PCA/ICA are not
dominated by high-power channels, subjects, or frequency bands.

**Reshaping** combines subjects and frequencies into the observation axis,
and channels and time into the feature axis:

```
(S, C, F, T)  →  transpose to  (S, F, C, T)
              →  reshape to     (S × F,  C × T)
                                observations  features
```

Each row of the resulting 2-D matrix is the z-scored power across all
channels and time points for a single subject at a single wavelet
frequency.  PCA/ICA will discover **spatial-temporal patterns** —
channel × time fingerprints — shared across subjects and frequencies.

### Why this reshape?

By treating each (subject, frequency) pair as an independent observation:

- We discover **channel × time patterns** that recur across both
  individuals and spectral bands — network activation modes.
- The observation-to-feature ratio (S×F vs C×T) determines the
  statistical regime of the decomposition.
- Scores can be reshaped to `(S, F, K)` to reveal which subjects
  and which frequency bands activate each spatial-temporal mode.


In [ ]:
# Z-score along time: each (subject, channel, frequency) slice → mean=0, std=1
bb_z = zscore_by_time(bb_data)  # (S, C, F, T)

# Reshape: (S, C, F, T) → transpose → (S, F, C, T) → (S*F, C*T)
bb_z_sf = bb_z.transpose(0, 2, 1, 3)  # (S, F, C, T)
n_obs = n_subjects * n_freqs
n_feat = n_channels * n_times
X_sf = bb_z_sf.reshape(n_obs, n_feat)  # (S*F, C*T)

print(f"Reshaped matrix shape : {X_sf.shape}")
print(f"  Observations (S\u00d7F)  : {X_sf.shape[0]}")
print(f"  Features     (C\u00d7T)  : {X_sf.shape[1]}")
print(f"Row means  \u2248 0 : {X_sf.mean(axis=1).mean():.6f}")
print(f"Row stds        : {X_sf.std(axis=1).mean():.4f}")

---
## Step 2 — PCA Dimensionality Reduction + ICA Decomposition

We first reduce the `C × T` feature space to `N_COMPONENTS_PCA`
principal components, keeping the directions of maximum variance.  Then
FastICA rotates the PCA subspace to maximise statistical independence,
yielding `N_COMPONENTS_ICA` independent components.

**Results:**

| Object | Shape | Description |
|--------|-------|-------------|
| `ica_scores` | `(S×F, K)` | Per-observation weight for each IC |
| `ica_components` | `(K, C×T)` | Spatial-temporal pattern of each IC |
| `scores_2d` | `(S, F, K)` | ICA scores reshaped to subject × frequency |
| `components_2d` | `(K, C, T)` | ICA components reshaped to channel × time |


In [ ]:
# --- PCA ---
pca = PCA(n_components=N_COMPONENTS_PCA, random_state=ICA_RANDOM_STATE)
pca_scores = pca.fit_transform(X_sf)  # (S*F, K_pca)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, len(explained) + 1), explained, color="steelblue")
axes[0].set_xlabel("Component")
axes[0].set_ylabel("Variance explained")
axes[0].set_title(f"PCA Scree Plot \u2014 {LABEL}")

axes[1].plot(range(1, len(cumulative) + 1), cumulative, "o-", color="coral")
axes[1].axhline(0.9, ls="--", color="gray", label="90%")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance explained")
axes[1].set_title(f"Cumulative Variance \u2014 {LABEL}")
axes[1].legend()

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "pca_scree.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

print(
    f"Top {N_COMPONENTS_PCA} components explain "
    f"{cumulative[-1] * 100:.1f}% of total variance."
)

# --- ICA ---
ica = FastICA(
    n_components=N_COMPONENTS_ICA,
    random_state=ICA_RANDOM_STATE,
    max_iter=500,
    whiten="unit-variance",
)
ica_scores = ica.fit_transform(pca_scores)  # (S*F, K_ica)
ica_components = ica.components_ @ pca.components_  # (K_ica, C*T)

# Reshape ICA scores to (S, F, K) for downstream analysis
scores_2d = ica_scores.reshape(n_subjects, n_freqs, N_COMPONENTS_ICA)  # (S, F, K)

# Reshape ICA components to (K, C, T) for spatial-temporal analysis
components_2d = ica_components.reshape(
    N_COMPONENTS_ICA, n_channels, n_times
)  # (K, C, T)

print(f"ICA scores shape       : {ica_scores.shape}")
print(f"ICA components shape   : {ica_components.shape}")
print(f"Scores 2-D shape       : {scores_2d.shape}  (S, F, K)")
print(f"Components 2-D shape   : {components_2d.shape}  (K, C, T)")

---
## Analysis (a) — Intersubject Correlation Matrix of ICA Components

For each ICA component we compute a **subject × subject** Pearson
correlation matrix.  Each subject is represented by their
**frequency loading vector** (shape `F`, i.e. the ICA score across
all wavelet frequencies).

High off-diagonal correlations indicate that the component has a
consistent spectral activation profile across individuals — a
hallmark of stimulus-driven (rather than noise-driven) modes.

### How this differs from combined-features (Approach 1)

In the combined-features notebook, each subject's loading vector spans
`C × F` (channels and frequencies).  Here the loading vector spans
only `F` (frequencies), because channels live in the component pattern
rather than the score.  This means the ISC matrix specifically
measures **spectral consistency** across subjects for each mode.


In [ ]:
# Per-subject frequency loading vector for each IC
# scores_2d: (S, F, K)
n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(
    1, n_show, figsize=(3.5 * n_show, 3.5), constrained_layout=True
)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    # Each subject's (F,) frequency-loading vector for IC i
    corr_mat = np.corrcoef(scores_2d[:, :, i])  # (S, S)
    im = ax.imshow(corr_mat, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(n_subjects))
    ax.set_yticks(range(n_subjects))
    ax.set_xticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig.suptitle(
    f"Intersubject Correlation of IC Freq Loadings \u2014 {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes[-1], label="Pearson r", shrink=0.8)
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "isc_component_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (b) — Component Temporal Profiles (Mean ± Std Across Subjects)

Each ICA component has a spatial-temporal pattern `(C, T)`.  To obtain
the **temporal profile** of each component, we average the absolute
component values across channels → `(T,)`.  This gives the overall
temporal envelope of the spatial-temporal mode.

To assess per-subject variability, we **project** each subject's data
through the component's channel pattern.  For subject *s* and component
*k*:

$$a_{s,k}(t) = \frac{1}{F}\sum_{f} \text{score}_{s,f,k} \;\cdot\; \frac{1}{C}\sum_{c} w_{k,c} \;\cdot\; z_{s,c,f}(t)$$

where $w_{k,c}$ is the channel weight of IC $k$ (averaged over time from
the component pattern) and $z_{s,c,f}(t)$ is the z-scored wavelet power.

We plot the **mean** across subjects with ± 1 std bands.

### How this differs from combined-features (Approach 1)

In the combined-features notebook, the temporal pattern IS the ICA
component (shape `(T,)`) and per-subject variability comes from ICA
scores.  Here, the component is a `(C, T)` spatial-temporal map, so
we need to marginalise over channels to recover the temporal profile.
The per-subject variability comes from projecting the original data
through both the ICA scores (which vary by subject and frequency)
and the component's channel weights.


In [ ]:
# Channel-averaged component temporal profiles: (K, T)
component_time_profiles = components_2d.mean(axis=1)  # (K, T)

# Per-subject temporal activations:
# For each subject s, component k:
#   weight the z-scored data by channel pattern and average over freqs
# Channel weights for each IC: time-averaged component pattern → (K, C)
channel_weights = components_2d.mean(axis=2)  # (K, C)

# Per-subject projection:
# bb_z: (S, C, F, T),  channel_weights: (K, C),  scores_2d: (S, F, K)
# Step 1: weight data by channel pattern → (S, K, F, T)
weighted_data = np.einsum("kc,scft->skft", channel_weights, bb_z)  # (S, K, F, T)
# Step 2: weight by frequency scores and average → (S, K, T)
subject_temporal = np.einsum("sfk,skft->skt", scores_2d, weighted_data) / (
    n_freqs * n_channels
)  # (S, K, T)

mean_temporal = subject_temporal.mean(axis=0)  # (K, T)
std_temporal = subject_temporal.std(axis=0)  # (K, T)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.5 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.plot(time, mean_temporal[i], lw=0.8, color="seagreen", label="mean")
    ax.fill_between(
        time,
        mean_temporal[i] - std_temporal[i],
        mean_temporal[i] + std_temporal[i],
        alpha=0.25,
        color="seagreen",
        label="\u00b1 1 std",
    )
    ax.set_ylabel(f"IC {i + 1}")
    ax.set_title(f"Component {i + 1} \u2014 Temporal Profile", fontsize=10)
    if i == 0:
        ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"ICA Component Temporal Profiles (mean \u00b1 std across subjects) \u2014 {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_temporal_profiles.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (c) — Frequency × Time Mean-Loading Heatmap

For each ICA component, we compute the **mean loading per subject**
at each wavelet frequency and time point.  The IC channel weights
are derived from the component `(K, C, T)` by averaging over time,
then used to project the z-scored data:

```
channel_weights(k, c) = mean_t[ components_2d(k, c, t) ]
weighted_data(s,k,f,t) = sum_c[ channel_weights(k,c) × bb_z(s,c,f,t) ]
loading(k, f, t) = (1/(S×C)) × sum_s[ scores_2d(s,f,k) × weighted_data(s,k,f,t) ]
```

This is equivalent to `mean_{s,c}[ scores(s,f,k) × ch_w(k,c) × bb_z(s,c,f,t) ]`.

### How this differs from combined-features (Approach 1)

Combined-features has a direct 4-D score tensor `(S,C,F,K)` to weight
against `bb_z`.  Here, the ICA scores are `(S,F,K)` and channels live
in the component; the heatmap combines both to reconstruct the effective
frequency × time activation.


In [ ]:
n_show = min(6, N_COMPONENTS_ICA)
# Channel weights from components: mean over time
channel_weights = components_2d.mean(axis=2)  # (K, C)
# Project bb_z through channel weights: sum over channels
weighted_data = np.einsum("kc,scft->skft", channel_weights, bb_z)  # (S,K,F,T)
# Combine with scores_2d and average over subjects and channels
ft_loading = np.einsum("sfk,skft->kft", scores_2d, weighted_data) / (
    n_subjects * n_channels
)

fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    data_i = ft_loading[i]  # (F, T)
    vmin_s, vmax_s = np.percentile(data_i, 1), np.percentile(data_i, 99)
    ax.pcolormesh(
        time,
        FREQS,
        data_i,
        cmap="inferno",
        vmin=vmin_s,
        vmax=vmax_s,
    )
    ax.set_ylabel("Freq (Hz)")
    ax.set_title(f"IC {i + 1} \u2014 Freq \u00d7 Time Mean Loading", fontsize=10)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Frequency \u00d7 Time Mean Loading per IC \u2014 {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_time_frequency.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")


---
## Analysis (d) — Mean and Variance of Component Channel Loadings as Topomaps

Channel loadings for each component are obtained from the ICA
component matrix by averaging over time → `(K, C)`.  These represent
the **spatial fingerprint** of each spatial-temporal mode.

To assess inter-individual variability, we combine the per-subject
frequency scores with the component channel weights to compute
per-subject channel loadings `(S, C, K)`, then compute:

- **Mean** across subjects → `(C, K)` — the average spatial distribution.
- **Variance** across subjects → `(C, K)` — electrodes where the mode
  strength varies most between individuals.

### How this differs from combined-features (Approach 1)

In the combined-features notebook, channel loadings come from ICA
**scores** (averaged over frequencies).  Here, the component pattern
itself contains the channel information (time-averaged from the
`(C, T)` component map), and per-subject variation comes from
weighting by each subject's frequency scores.


In [ ]:
# Component channel loadings: time-averaged → (K, C)
comp_channel_loadings = components_2d.mean(axis=2)  # (K, C)

# Per-subject channel loadings:
# Weight component channel pattern by each subject's mean frequency score
# scores_2d: (S, F, K) → mean over F → (S, K)
subject_mean_scores = scores_2d.mean(axis=1)  # (S, K)
# Per-subject channel loading: (S, K) × (K, C) → (S, C, K)
ica_channel_loadings = np.einsum(
    "sk,kc->sck", subject_mean_scores, comp_channel_loadings
)  # (S, C, K)

# Mean and variance across subjects
ica_ch_mean = ica_channel_loadings.mean(axis=0)  # (C, K)
ica_ch_var = ica_channel_loadings.var(axis=0)  # (C, K)

# Get MNE Info for topomap
info = analyzers[LABEL].info
info = mne.pick_info(info, mne.pick_types(info, eeg=True))
if n_channels < len(info.ch_names):
    info = mne.pick_info(info, list(range(n_channels)))

n_show = min(6, N_COMPONENTS_ICA)

# --- Mean topomaps ---
_vlim_mean = np.percentile(np.abs(ica_ch_mean[:, :n_show]), 99)
fig_mean, axes_mean = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
if n_show == 1:
    axes_mean = [axes_mean]

for i, ax in enumerate(axes_mean):
    im, _ = plot_topomap(
        ica_ch_mean[:, i],
        info,
        axes=ax,
        show=False,
        cmap="RdBu_r",
        vlim=(-_vlim_mean, _vlim_mean),
    )
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig_mean.suptitle(
    f"Mean Component Channel Loading (topomap) \u2014 {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes_mean[-1], label="mean loading")
fig_mean.tight_layout()
if SAVE_PLOTS:
    fig_mean.savefig(PLOTS_DIR / "ica_topomap_mean.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

# --- Variance topomaps ---
_vmax_var = np.percentile(ica_ch_var[:, :n_show], 99)
fig_var, axes_var = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
if n_show == 1:
    axes_var = [axes_var]

for i, ax in enumerate(axes_var):
    im, _ = plot_topomap(
        ica_ch_var[:, i],
        info,
        axes=ax,
        show=False,
        cmap="YlOrRd",
        vlim=(0, _vmax_var),
    )
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig_var.suptitle(
    f"Variance of Component Channel Loading (topomap) \u2014 {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes_var[-1], label="variance")
fig_var.tight_layout()
if SAVE_PLOTS:
    fig_var.savefig(
        PLOTS_DIR / "ica_topomap_variance.png", dpi=150, bbox_inches="tight"
    )
plt.show()
plt.close("all")

---
## Analysis (e) — Per-Subject Loading Bar Plot for Each Component

For each ICA component, compute the **mean absolute score** across
frequencies per subject.  This scalar summarises how strongly each
participant expresses the spatial-temporal mode across all frequency
bands.

Subjects with uniformly high loadings indicate a stimulus-driven mode;
uneven loadings may reflect individual differences.

### How this differs from combined-features (Approach 1)

In the combined-features notebook, subject loadings are the mean
absolute ICA score over channels AND frequencies.  Here, channels
live in the component pattern, so subject loadings are the mean
absolute score over frequencies only.  This is a more focused
measure of each subject's spectral engagement with the mode.


In [ ]:
# Subject loadings: mean |score| over frequencies
subject_loadings = np.abs(scores_2d).mean(axis=1)  # (S, K)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 4), sharey=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.barh(
        range(n_subjects),
        subject_loadings[:, i],
        color="seagreen",
    )
    ax.set_yticks(range(n_subjects))
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=8)
    ax.set_xlabel("|score|")
    ax.set_title(f"IC {i + 1}", fontsize=10)

axes[0].set_ylabel("Subject")
fig.suptitle(
    f"Per-Subject Loading per Component \u2014 {LABEL}",
    fontsize=13,
    y=1.02,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_subject_loadings.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (f) — IC Temporal Patterns (Component Waveforms)

Each ICA component is a spatial-temporal pattern of shape `(C, T)`.
To obtain the **temporal waveform** of each component, we average
over channels → `(T,)`. This gives the shared temporal structure
extracted by ICA across the spatial-temporal mode.

**Interpretation:** Peaks and troughs indicate moments where the
corresponding channel × time pattern is strongly active. Unlike
analysis (b) which shows per-subject *weighted* activations, this
is the component waveform itself (channel-averaged).

### How this differs from combined-features (Approach 1)

In the combined-features notebook, the ICA component IS directly
a temporal pattern `(T,)` and we plot it directly. Here the
component is `(C, T)`, so we need to average over channels to
recover the temporal profile. The result is `component_time_profiles`
which is already computed in analysis (b).

In [ ]:
# Channel-averaged component temporal profiles: (K, T)
# (already computed as component_time_profiles in cell above)
n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.0 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.plot(time, component_time_profiles[i], lw=0.6, color="teal")
    ax.set_ylabel(f"IC {i + 1}")
    ax.set_title(f"Component {i + 1} — Temporal Waveform", fontsize=10)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"ICA Component Temporal Patterns — {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_component_timecourse.png", dpi=150, bbox_inches="tight"
    )
plt.show()
plt.close("all")

---
## Analysis (g) — Per-Subject Per-Component Loading Heatmap

A 2-D heatmap with subjects on the y-axis and ICA components on the x-axis.
Each cell shows the mean |loading| for that subject–component pair, averaged
over frequencies.

**Interpretation:** Rows that are uniformly bright indicate subjects whose data
strongly participates in all modes; columns that are bright indicate components
that are strongly expressed across all subjects.

### How this differs from combined-features (Approach 1)

In combined-features, the heatmap averages `|scores_4d|` over both channels and
frequencies. Here, channels live in the component pattern, so we average
`|scores_2d|` over frequencies only. This gives a focused view of each subject's
spectral engagement with each spatial-temporal mode.

In [ ]:
# Mean |loading| across frequencies for each subject × IC
subject_loadings_hm = np.abs(scores_2d).mean(axis=1)  # (S, K)

fig, ax = plt.subplots(
    figsize=(max(8, N_COMPONENTS_ICA * 0.8), max(4, n_subjects * 0.4))
)
im = ax.imshow(subject_loadings_hm, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(N_COMPONENTS_ICA))
ax.set_xticklabels([f"IC {k + 1}" for k in range(N_COMPONENTS_ICA)], fontsize=9)
ax.set_yticks(range(n_subjects))
ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=9)
ax.set_xlabel("Component")
ax.set_ylabel("Subject")
ax.set_title(f"Per-Subject Per-Component Loading Heatmap — {LABEL}", fontsize=13)
plt.colorbar(im, ax=ax, label="mean |loading|")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_subject_component_heatmap.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

---
## Analysis (h) — Subject-Consistency (Correlation) per Component

For each IC, compute the S×S intersubject correlation matrix (same as analysis a),
then extract the mean of the upper-triangle off-diagonal entries. This gives a
single consistency score per IC, plotted as a bar chart.

**Interpretation:** High bars indicate components that are consistently expressed
across subjects (shared neural response); low/negative bars indicate
subject-specific or noisy components.

### How this differs from combined-features (Approach 1)

In combined-features, the per-subject loading vector spans `C × F`. Here it
spans only `F` (frequencies), because channels live in the component. The
consistency measure specifically quantifies **spectral consistency** across
subjects for each mode.

In [ ]:
# Per-subject frequency loading vector for each IC: scores_2d[:, :, k] → (S, F)
isc_per_ic = np.zeros(N_COMPONENTS_ICA)
for k in range(N_COMPONENTS_ICA):
    corr_mat = np.corrcoef(scores_2d[:, :, k])  # (S, S)
    # Mean of upper-triangle off-diagonal entries
    triu_idx = np.triu_indices(n_subjects, k=1)
    isc_per_ic[k] = corr_mat[triu_idx].mean()

fig, ax = plt.subplots(figsize=(max(8, N_COMPONENTS_ICA * 0.7), 4))
colors = ["steelblue" if v >= 0 else "salmon" for v in isc_per_ic]
ax.bar(range(1, N_COMPONENTS_ICA + 1), isc_per_ic, color=colors)
ax.set_xlabel("Component")
ax.set_ylabel("Mean pairwise ISC (Pearson r)")
ax.set_xticks(range(1, N_COMPONENTS_ICA + 1))
ax.axhline(0, color="gray", ls="--", lw=0.8)
ax.set_title(f"Subject-Consistency per IC — {LABEL}", fontsize=13)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_subject_consistency_bar.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

# Print sorted consistency values
print("Subject-consistency (mean pairwise ISC) per IC:")
for k in np.argsort(isc_per_ic)[::-1]:
    print(f"  IC {k + 1}: {isc_per_ic[k]:.4f}")

---
## Analysis (i) — Frequency Profile per Component

For each IC, compute the mean |loading| within each canonical frequency band
(delta 1–4 Hz, theta 4–8, alpha 8–13, beta 13–30, gamma 30–70). This reveals
which frequency band dominates each component.

**Interpretation:** A component dominated by alpha means its spatial–temporal
pattern captures alpha-range wavelet power co-fluctuations. Components with
flat profiles span multiple bands equally.

### How this differs from combined-features (Approach 1)

In combined-features, the frequency axis lives in `scores_4d` and is averaged
over subjects and channels. Here, the frequency axis is directly in `scores_2d`
(averaged over subjects only). The result is the same type of profile but
derived from a 3-D rather than 4-D score tensor.

In [ ]:
# Assign each wavelet frequency to its canonical band
band_names = list(FREQUENCY_BANDS.keys())
band_ranges = list(FREQUENCY_BANDS.values())
n_bands = len(band_names)

# Mean |loading| per frequency per IC: average |scores_2d| over subjects
freq_profile = np.abs(scores_2d).mean(axis=0)  # (F, K)

band_profile = np.zeros((n_bands, N_COMPONENTS_ICA))
for b_idx, (lo, hi) in enumerate(band_ranges):
    mask = (FREQS >= lo) & (FREQS < hi)
    if mask.sum() > 0:
        band_profile[b_idx] = freq_profile[mask].mean(axis=0)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 4), sharey=True)
if n_show == 1:
    axes = [axes]

band_colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
for i, ax in enumerate(axes):
    ax.barh(
        range(n_bands),
        band_profile[:, i],
        color=band_colors[:n_bands],
    )
    ax.set_yticks(range(n_bands))
    ax.set_yticklabels(band_names, fontsize=9)
    ax.set_xlabel("mean |loading|")
    ax.set_title(f"IC {i + 1}", fontsize=10)

axes[0].set_ylabel("Frequency band")
fig.suptitle(
    f"Frequency Profile per Component — {LABEL}",
    fontsize=13,
    y=1.02,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_frequency_profile.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Summary

### Decomposition overview

This notebook uses the reshape `(S × F, C × T)` — subjects and
frequencies as observations, channels and time as features.  ICA
discovers **spatial-temporal modes** (channel × time patterns) that
recur across subjects and frequency bands.

| Aspect | Combined Features (Approach 1) | Subject–Freq Features (here) |
|--------|--------------------------------|------------------------------|
| Observations | S × C × F | S × F |
| Features | T | C × T |
| Components represent | Temporal patterns `(T,)` | Spatial-temporal modes `(C, T)` |
| Scores represent | Per-triplet weights `(S, C, F, K)` | Per-subject-freq weights `(S, F, K)` |
| Channel info lives in | Scores (axis 1) | Components (axis 1) |
| ISC measures | Spectral-spatial consistency | Spectral consistency |

The key advantage of this reshape is that channel and time appear
together in the component pattern, allowing ICA to find **network-level
activation fingerprints** — coordinated spatial-temporal patterns that
generalise across both subjects and frequency bands.

### Variables available for further analysis

| Variable | Shape | Description |
|----------|-------|-------------|
| `bb_z` | `(S, C, F, T)` | Z-scored 4-D wavelet power tensor |
| `X_sf` | `(S×F, C×T)` | Z-scored reshaped 2-D matrix |
| `pca` | — | Fitted PCA object |
| `pca_scores` | `(S×F, K_pca)` | PCA-transformed scores |
| `ica` | — | Fitted FastICA object |
| `ica_scores` | `(S×F, K_ica)` | ICA scores (per-observation weights) |
| `ica_components` | `(K_ica, C×T)` | ICA spatial-temporal component patterns |
| `scores_2d` | `(S, F, K)` | ICA scores reshaped to subject × frequency |
| `components_2d` | `(K, C, T)` | ICA components reshaped to channel × time |
| `component_time_profiles` | `(K, T)` | Channel-averaged component temporal profiles |
| `subject_temporal` | `(S, K, T)` | Per-subject temporal activations |
| `ica_channel_loadings` | `(S, C, K)` | Per-subject channel loadings |

### Analyses implemented

| # | Analysis | Key finding |
|---|----------|-------------|
| (a) | Intersubject correlation matrix | Which modes have consistent freq profiles across subjects |
| (b) | Temporal profiles (mean ± std) | When each mode is active and how variable across subjects |
| (c) | Freq × Time mean loading | Mean IC loading at each wavelet frequency and time |
| (d) | Mean / variance topomaps | Spatial distribution and inter-individual variability |
| (e) | Per-subject loading bars | Individual-level contribution to each mode |
| (f) | IC temporal patterns | Channel-averaged component waveforms — shared temporal structure |
| (g) | Subject × Component heatmap | Per-subject per-component loading strength (over frequencies) |
| (h) | Subject-consistency bar plot | Mean pairwise ISC per component — one bar per IC |
| (i) | Frequency profile per component | Which canonical band dominates each IC |

See `README.md` in this directory for the full analysis rationale,
alternative decomposition strategies, and ideas for future extensions.